In [ ]:
pip install tensorflow


In [ ]:
pip show tensorflow

Name: tensorflow
Version: 2.15.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.10/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, setuptools, six, tensorboard, tensorflow-estimator, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: dopamine-rl, tf_keras


In [ ]:
pip install optree

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 6.7 MB/s eta 0:00:00


In [ ]:
import sys
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import optimizers

import tensorflow as tf; tf.keras


from tensorflow.keras.layers import Dense, Input,Dropout, Flatten, Activation,Convolution2D, MaxPooling2D
from tensorflow.keras import Sequential
from tensorflow.keras.activations import sigmoid


#from tensorflow.python.keras.models import Sequential
#from tensorflow.python.keras.layers import Dropout, Flatten, Dense, Activation
#from tensorflow.python.keras.layers import  Convolution2D, MaxPooling2D
from tensorflow.python.keras import backend as K


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
K.clear_session()



data_entrenamiento = './drive/MyDrive/data1/entrenamiento'
data_validacion = './drive/MyDrive/data1/validacion'


In [ ]:
epocas=2000
longitud, altura = 150, 150
batch_size = 32
pasos = 1
validation_steps = 300
filtrosConv1 = 32
filtrosConv2 = 64
tamano_filtro1 = (3, 3)
tamano_filtro2 = (2, 2)
tamano_pool = (2, 2)
clases = 2
lr = 0.0004

In [ ]:
##Preparamos nuestras imagenes

entrenamiento_datagen = ImageDataGenerator(
    rescale=1. / 255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1. / 255)

entrenamiento_generador = entrenamiento_datagen.flow_from_directory(
    data_entrenamiento,
    target_size=(altura, longitud),
    batch_size=batch_size,
    class_mode='categorical')

validacion_generador = test_datagen.flow_from_directory(
    data_validacion,
    target_size=(altura, longitud),
    batch_size=batch_size,
    class_mode='categorical')

Found 80 images belonging to 2 classes.
Found 80 images belonging to 2 classes.


In [ ]:
cnn = Sequential()
cnn.add(Convolution2D(filtrosConv1, tamano_filtro1, padding ="same", input_shape=(longitud, altura, 3), activation='relu'))
cnn.add(MaxPooling2D(pool_size=tamano_pool))

cnn.add(Convolution2D(filtrosConv2, tamano_filtro2, padding ="same"))
cnn.add(MaxPooling2D(pool_size=tamano_pool))

cnn.add(Flatten())
cnn.add(Dense(256, activation='relu'))
cnn.add(Dropout(0.5))
cnn.add(Dense(clases, activation='softmax'))

cnn.compile(loss='categorical_crossentropy',
            optimizer='adam',
            metrics=['accuracy'])

In [ ]:
cnn.fit(
    entrenamiento_generador,
    steps_per_epoch=pasos,
    epochs=epocas,
    validation_data=validacion_generador,
    validation_steps=validation_steps)


Epoch 1/2000
1/1 [==============================] - ETA: 0s - loss: 0.6612 - accuracy: 0.6875

1/1 [==============================] - 69s 69s/step - loss: 0.6612 - accuracy: 0.6875 - val_loss: 1.4855 - val_accuracy: 0.5125
Epoch 2/2000
1/1 [==============================] - 1s 785ms/step - loss: 3.1240 - accuracy: 0.4375
Epoch 3/2000
1/1 [==============================] - 0s 314ms/step - loss: 7.6403 - accuracy: 0.5625
Epoch 4/2000
1/1 [==============================] - 0s 190ms/step - loss: 2.4392 - accuracy: 0.5625
Epoch 5/2000
1/1 [==============================] - 0s 336ms/step - loss: 5.4728 - accuracy: 0.5312
Epoch 6/2000
1/1 [==============================] - 0s 498ms/step - loss: 2.2937 - accuracy: 0.5625
Epoch 7/2000
1/1 [==============================] - 1s 506ms/step - loss: 4.1272 - accuracy: 0.4688
Epoch 8/2000
1/1 [==============================] - 0s 495ms/step - loss: 2.6788 - accuracy: 0.5000
Epoch 9/2000
1/1 [==============================] - 0s 261ms/step - loss: 1.3546 - accuracy: 0.5000
Epoch 10/2000
1/1 [==============================] - 0s 339ms/step - los

In [ ]:
target_dir = './modelo/'
if not os.path.exists(target_dir):
  os.mkdir(target_dir)
cnn.save('./modelo/modelo.h5')
cnn.save_weights('./modelo/pesos.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
def predict(file):

  x = load_img(file, target_size=(longitud, altura))

  x = img_to_array(x)

  x = np.expand_dims(x, axis=0)

  array = cnn.predict(x)
  print(array)
  result = array[0]
  print(result)
  answer = np.argmax(result)
  if answer == 0:
    print("pred: Gato")
  elif answer == 1:
    print("pred: Perro")

  return answer

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from keras.models import load_model


file='./drive/MyDrive/data1/gato1.jpg'
predict(file)
x = load_img(file, target_size=(longitud, altura))
x = img_to_array(x)
x = np.expand_dims(x, axis=0)
array = cnn.predict(x)




1/1 [==============================] - 0s 242ms/step
[[0. 1.]]
[0. 1.]
pred: Perro
1/1 [==============================] - 0s 22ms/step


In [ ]:
target_dir = './modelo/'

longitud, altura = 150, 150
modelo = './modelo/modelo.h5'
pesos_modelo = './modelo/pesos.h5'
cnn1 = load_model(modelo)
cnn1.load_weights(pesos_modelo)

In [ ]:
def predict1(file):

  x = load_img(file, target_size=(longitud, altura))

  x = img_to_array(x)

  x = np.expand_dims(x, axis=0)

  array = cnn1.predict(x)
  print(array)
  result = array[0]
  print(result)
  answer = np.argmax(result)
  if answer == 1:
    print("pred: Perro")
  elif answer == 0:
    print("pred: Gato")

  return answer

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from keras.models import load_model


file='./drive/MyDrive/data1/dog28.jpg'
predict(file)
x = load_img(file, target_size=(longitud, altura))
x = img_to_array(x)
x = np.expand_dims(x, axis=0)
array = cnn.predict(x)


1/1 [==============================] - 0s 18ms/step
[[0. 1.]]
[0. 1.]
pred: Perro
1/1 [==============================] - 0s 21ms/step
